<div style="
    width:100%;
    max-width:1080px;
    margin:24px 0 32px 0;
    background:linear-gradient(180deg, #1f2937 0%, #0f172a 100%);
    border:1px solid #334155;
    border-radius:22px;
    box-shadow:0 18px 38px rgba(15,23,42,0.24);
    overflow:hidden;
    color:#e6edf7;
">

<div style="
    height:8px;
    background:linear-gradient(90deg, #c2410c 0%, #f59e0b 100%);
"></div>

<div style="
    max-width:860px;
    padding:34px 38px 36px 38px;
    margin-left:0;
    margin-right:auto;
    font-family:'Aptos', 'Segoe UI Variable', 'Segoe UI', sans-serif;
    font-size:19px;
    line-height:1.78;
    color:#e6edf7;
">

<div style="margin-bottom:22px;">
<h1 style="margin:0 0 14px 0; font-size:34px; line-height:1.2; font-weight:700; color:#ffffff;">La Quantization: Visión General</h1>
<p style="margin:0; color:#cbd5e1; font-size:18px;">La quantization reduce precisión numérica para ganar eficiencia en inferencia. Este notebook introduce esa idea como una de las estrategias más conocidas para aligerar modelos.</p>
</div>

<div style="display:grid; gap:14px;">
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Por qué importa</div>
    <div style="color:#f8fafc;">Puede disminuir tamaño, latencia y consumo de recursos, algo especialmente valioso en despliegues exigentes.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Qué vas a aprender aquí</div>
    <div style="color:#f8fafc;">A entender qué persigue la quantization y cuál es el equilibrio que propone entre eficiencia y fidelidad.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Regla práctica que debes recordar</div>
    <div style="color:#f8fafc;">Optimizar no consiste en recortar a ciegas; hay que medir el impacto real sobre el comportamiento del modelo.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Contexto mínimo</div>
    <div style="color:#f8fafc;">Quantizar implica representar pesos o activaciones con menor precisión para abaratar cálculo y memoria.</div>
  </div>
</div>

</div>

</div>

<hr style="border:0; border-top:1px solid #64748b; opacity:0.22; margin:36px 0 42px 0;">

In [ ]:
# Bloque 1 — Carga del modelo y del tokenizer
"""Este bloque carga un modelo preentrenado y su tokenizer asociado.
Este bloque importa "AutoModelForSequenceClassification" y "AutoTokenizer" de "transformers" para cargar un modelo preentrenado 
y su tokenizador correspondiente.

Aplicamos la clase "AutoModelForSequenceClassification", acompañada del método ".from_pretrained()", a la variable "modelo" para cargar un modelo preentrenado.
El argumento "bert-base-uncased" especifica el modelo preentrenado que se desea utilizar.
El parámetro "num_labels=" especifica el número de etiquetas de clasificación que el modelo debe predecir.
La variable "modelo" almacena una instancia del modelo para clasificación de secuencias. Su cabeza de clasificación se inicializa de nuevo y no se entrena en este ejemplo.

Aplicamos la clase "AutoTokenizer", acompañada del método ".from_pretrained()", a la variable "tokenizador" para cargar un tokenizador preentrenado.
El argumento "bert-base-uncased" especifica el modelo preentrenado que se desea utilizar.
La variable "tokenizador" almacena una instancia del tokenizador preentrenado lista para ser utilizada en tareas de clasificación de secuencias.
Estos son los componentes utilizados para el ejemplo de cuantización.
El modelo y el tokenizador están preentrenados para inglés, mientras que el texto del ejemplo está en español."""

from transformers import AutoModelForSequenceClassification, AutoTokenizer

modelo = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)
tokenizador = AutoTokenizer.from_pretrained("bert-base-uncased")

modelo, tokenizador # Muestra el modelo y el tokenizador cargados.


In [ ]:
# Bloque 2 — Preparación de una entrada para inferencia
"""Este bloque define una entrada de texto para evaluar el comportamiento del modelo antes y después de la cuantización.
Definimos un texto de entrada para el modelo.
La variable "texto_entrada" contiene un texto para evaluar el modelo en inferencia.

Asignamos la llamada a la instancia de la clase "tokenizador()" a la variable "entrada_tokenizada" mediante la expresión "tokenizador(texto_entrada, return_tensors="pt")".
El argumento "texto_entrada" indica el texto que se desea tokenizar.
El parámetro "return_tensors=" indica el tipo de tensor que se desea obtener, en este caso, "pt" para PyTorch.
La variable "entrada_tokenizada" almacena el texto de entrada tokenizado en formato tensor de PyTorch."""

texto_entrada = "El servicio fue excelente."

entrada_tokenizada = tokenizador(texto_entrada, return_tensors="pt")

entrada_tokenizada # Muestra el texto de entrada tokenizado en formato tensor de PyTorch.


In [ ]:
# Bloque 3 — Inferencia estándar
"""Este bloque ejecuta inferencia estándar sin cuantización.
Asignamos el atributo ".logits" de la salida de "modelo()" a la variable "logits_estandar".
El argumento "**entrada_tokenizada" indica los tensores tokenizados que se pasan al modelo.
Los asteriscos dobles "**" se utilizan para desempaquetar el diccionario "entrada_tokenizada" y pasar sus elementos como argumentos al modelo.
La variable "logits_estandar" contiene las salidas del modelo antes de realizar la operación argmax."""

logits_estandar = modelo(**entrada_tokenizada).logits

logits_estandar # Muestra las salidas del modelo antes de aplicar argmax.


In [ ]:
# Bloque 4 — Activación del modo no_grad
"""Este bloque activa el modo no_grad para reducir el uso de memoria y, según el caso, acelerar la inferencia.
Este bloque importa "torch" para trabajar con tensores en PyTorch y habilitar el modo no_grad durante la inferencia.

Aplicamos la clase "torch.no_grad()" mediante un bloque "with" para crear un entorno de gradientes desactivados durante la inferencia.
Dentro del entorno no_grad, asignamos el atributo ".logits" de la salida de "modelo()" a la variable "logits_no_grad".
El argumento "**entrada_tokenizada" indica los tensores tokenizados que se pasan al modelo en el entorno no_grad.
Los asteriscos dobles "**" se utilizan para desempaquetar el diccionario "entrada_tokenizada" y pasar sus elementos como argumentos al modelo.
La variable "logits_no_grad" contiene las salidas del modelo sin cálculo de gradientes."""

import torch

with torch.no_grad():
    logits_no_grad = modelo(**entrada_tokenizada).logits

logits_no_grad # Muestra las salidas del modelo sin cálculo de gradientes.


In [ ]:
# Bloque 5 — Cuantización dinámica del modelo
"""Este bloque aplica cuantización dinámica al modelo con el objetivo de reducir su tamaño y mejorar la inferencia.
Este bloque importa "copy" para crear una copia del modelo en memoria antes de aplicar la cuantización dinámica.
Este bloque importa "Int8DynamicActivationInt8WeightConfig" y "quantize_" de "torchao.quantization" para aplicar la cuantización dinámica al modelo.

Aplicamos la función "copy.deepcopy()" al modelo para crear una copia en memoria antes de aplicar la cuantización dinámica.
La variable "modelo_cuantizado" almacena la copia del modelo que será optimizada mediante cuantización dinámica.

Aplicamos la función "quantize_()" al modelo copiado y a la clase "Int8DynamicActivationInt8WeightConfig()" para realizar la cuantización dinámica.
Esta operación cuantiza las capas compatibles del modelo copiado sin modificar el modelo original; su efecto en el rendimiento depende del entorno.
La variable "modelo_cuantizado" almacena ahora la versión cuantizada del modelo copiado, lista para inferencia."""

import copy
from torchao.quantization import (Int8DynamicActivationInt8WeightConfig, quantize_,)

modelo_cuantizado = copy.deepcopy(modelo)
quantize_(modelo_cuantizado, Int8DynamicActivationInt8WeightConfig())

# Esta operación cuantiza las capas compatibles del modelo copiado sin modificar el modelo original.


In [ ]:
# Bloque 6 — Inferencia con modelo cuantizado
"""Este bloque ejecuta inferencia utilizando el modelo cuantizado.
Aplicamos la clase "torch.no_grad()" mediante un bloque "with" para crear un entorno de gradientes desactivados durante la inferencia.

Dentro del entorno no_grad, asignamos el atributo ".logits" de la salida de "modelo_cuantizado()" a la variable "logits_cuantizado".
El argumento "**entrada_tokenizada" indica los tensores tokenizados que se pasan al modelo cuantizado durante la inferencia.
Los asteriscos dobles "**" se utilizan para desempaquetar el diccionario "entrada_tokenizada" y pasar sus elementos como argumentos al modelo.
La variable "logits_cuantizado" contiene las salidas del modelo optimizado mediante cuantización."""

with torch.no_grad():
    logits_cuantizado = modelo_cuantizado(**entrada_tokenizada).logits

logits_cuantizado # Muestra las salidas del modelo cuantizado.


In [ ]:
# Bloque 7 — Comparación de tamaños del modelo
"""Este bloque compara el tamaño en bytes de los pesos serializados con PyTorch.
Este bloque importa "tempfile" para crear archivos temporales donde se guardan los pesos del modelo.
Este bloque importa "torch" para trabajar con tensores y guardar los pesos del modelo.

Definimos la función "def tamaño_modelo(modelo)" que calcula el tamaño en bytes del estado serializado de un modelo dado.
El argumento "modelo" es el modelo de PyTorch cuyo tamaño de pesos queremos calcular.

Creamos un entorno temporal para guardar los pesos del modelo y calcular su tamaño.
La expresión "with tempfile.TemporaryFile() as archivo:" crea un archivo temporal llamado "archivo" donde se guardan los pesos del modelo.

Dentro del bloque "with", guardamos los pesos del modelo en el archivo temporal y luego obtenemos el tamaño en bytes del archivo.
La expresión "torch.save(modelo.state_dict(), archivo)" guarda los pesos del modelo en el archivo temporal.

Retornamos el tamaño en bytes del archivo temporal usando "return archivo.tell()".
Los archivos temporales se eliminan al terminar; no se guardan modelos para su distribución.

Aplicamos la función "tamaño_modelo" a los modelos original y cuantizado para obtener sus tamaños en bytes correspondientes.
En cada caso, las variables "tamaño_original" y "tamaño_cuantizado" almacenan los tamaños serializados de los modelos original y cuantizado, respectivamente.
Esta comparación no mide directamente la memoria utilizada durante la inferencia."""

import tempfile
import torch

def tamaño_modelo(modelo):
    with tempfile.TemporaryFile() as archivo:
        torch.save(modelo.state_dict(), archivo)
        return archivo.tell()

tamaño_original = tamaño_modelo(modelo)
tamaño_cuantizado = tamaño_modelo(modelo_cuantizado)

tamaño_original, tamaño_cuantizado # Muestra los tamaños serializados en bytes de los modelos original y cuantizado.


In [ ]:
# Bloque 8 — Resultados finales
"""Este bloque organiza los resultados del proceso de cuantización.
La variable resultado contiene logits estándar, logits sin gradientes, logits cuantizados y tamaños serializados del modelo."""

resultado = {
    "logits_estandar": logits_estandar,
    "logits_no_grad": logits_no_grad,
    "logits_cuantizado": logits_cuantizado,
    "tamaño_original_bytes": tamaño_original,
    "tamaño_cuantizado_bytes": tamaño_cuantizado
}

resultado # Muestra el diccionario con los resultados finales del proceso de cuantización.


<div style="
    width:100%;
    max-width:1080px;
    margin:40px 0 24px 0;
    background:linear-gradient(180deg, #1f2937 0%, #0f172a 100%);
    border:1px solid #334155;
    border-radius:22px;
    box-shadow:0 18px 38px rgba(15,23,42,0.24);
    overflow:hidden;
    color:#e6edf7;
">

<div style="
    height:8px;
    background:linear-gradient(90deg, #0f766e 0%, #2dd4bf 100%);
"></div>

<div style="
    max-width:860px;
    padding:34px 38px 36px 38px;
    margin-left:0;
    margin-right:auto;
    font-family:'Aptos', 'Segoe UI Variable', 'Segoe UI', sans-serif;
    font-size:19px;
    line-height:1.78;
    color:#e6edf7;
">

<div style="margin-bottom:22px;">
<h2 style="margin:0 0 14px 0; font-size:30px; line-height:1.2; font-weight:700; color:#ffffff;">Conclusiones</h2>
<p style="margin:0; color:#cbd5e1; font-size:18px;">En este notebook has visto la quantization como una estrategia para hacer más liviano el uso del modelo durante inferencia.</p>
</div>

<div style="display:grid; gap:14px;">
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Qué conviene recordar</div>
    <div style="color:#f8fafc;">Toda mejora de eficiencia debe evaluarse junto con la posible pérdida de calidad del sistema.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Error frecuente a evitar</div>
    <div style="color:#f8fafc;">Aplicar reducción de precisión sin medir cómo cambia el comportamiento real del modelo.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Siguiente paso natural</div>
    <div style="color:#f8fafc;">Detenerse en las Consideraciones de Rendimiento para dar continuidad directa al hilo del temario.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Conexión con el siguiente concepto</div>
    <div style="color:#f8fafc;">El siguiente notebook se centra en las Consideraciones de Rendimiento y amplía de forma natural lo que acabas de trabajar.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Siguiente notebook del temario</div>
    <div style="color:#f8fafc;">08_Inferencia_optimizada/04_Las_consideraciones_de_rendimiento.ipynb</div>
  </div>
</div>

</div>

</div>

<hr style="border:0; border-top:1px solid #64748b; opacity:0.22; margin:42px 0 24px 0;">